# Classificação de tópicos predefinidos: 54 classes do AAPD

Em vez de descobrir tópicos e nomeá-los depois, começamos com um catálogo de **54 classes**, descrições e palavras-semente. Usamos os **2.000 resumos** do experimento anterior.

**Seeded LDA:** cada classe tem seu tópico, com palavras favorecidas por um prior persistente.

**Guided BERTopic:** as sementes orientam a descoberta; depois uma projeção semântica associa os clusters aprendidos às classes do catálogo. Não há garantia de 54 clusters.

Os rótulos reais não são usados para aprender tópicos. Servem para escolher o número de classes atribuídas na validação e avaliar o teste.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display
import plotly.express as px

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'dataset_loader.py').exists())
EXPERIMENT_DIR = ROOT / 'experimento_classificacao_topicos'
sys.path.insert(0, str(EXPERIMENT_DIR))
from topicos_predefinidos import CLASSES, DESCRIPTIONS, SEED_TOPICS
from experimento import load_data, run

pd.set_option('display.max_colwidth', 100)
df, Y, split = load_data()
print(f'{len(df)} resumos, {Y.shape[1]} classes; média de {Y.sum(1).mean():.2f} classes/artigo')
display(pd.DataFrame({'Classe': CLASSES, 'Descrição': DESCRIPTIONS,
                      'Sementes': [', '.join(row) for row in SEED_TOPICS]}))

## 1. Separar treino, validação e teste

Treino aprende os modelos com 1.200 artigos; validação escolhe `k` (1 a 5 rótulos) com 300 artigos; teste mede o resultado em 500 artigos. O split é fixo e igual para os dois métodos. Nenhuma classe rara é removida.

In [ ]:
support = pd.DataFrame({'Classe': CLASSES, **{name: Y[idx].sum(0) for name, idx in split.items()}})
display(support)
display(df[['doc_id', 'classes_reais_aapd', 'resumo_texto']].head(3))

## 2. Treinar os dois métodos e atribuir classes

As sementes estão em `topicos_predefinidos.py` e podem ser editadas. O Seeded LDA soma o prior das sementes em cada atualização, não apenas na inicialização.

O Guided BERTopic utiliza `seed_topic_list`, UMAP e HDBSCAN. Comparamos os descritores das classes com as palavras dos clusters aprendidos para criar a projeção. Classes sem cluster ficam com score zero; clusters com descritores empatados dividem o peso. O teste não participa desse mapeamento.

A célula pode levar alguns minutos. O modelo de embeddings roda em CPU e pode ser baixado no primeiro uso. Para usar apenas o cache, defina `LOCAL_FILES_ONLY = True`.

In [ ]:
LOCAL_FILES_ONLY = False
ITERATIONS = 20
SEED_STRENGTH = 20.0
MIN_CLUSTER_SIZE = 15
OUTPUT = EXPERIMENT_DIR / 'resultados' / 'notebook'
metrics, per_class, models = run(
    method='ambos', output=OUTPUT, iterations=ITERATIONS,
    seed_strength=SEED_STRENGTH, local_files_only=LOCAL_FILES_ONLY,
    min_cluster_size=MIN_CLUSTER_SIZE,
)
display(metrics.round(4))

## 3. Conferir se a orientação funcionou

No LDA, observe a cobertura das sementes e as palavras aprendidas para cada classe. No BERTopic, confira quantos clusters surgiram e quais classes receberam seus pesos. A massa de pertinência pode ser menor que 1, pois há rejeição/outliers. Os scores dos dois métodos não são probabilidades calibradas comparáveis.

In [ ]:
seeded = models['Seeded LDA']
guided = models['Guided BERTopic']
display(seeded.seed_coverage_)
display(seeded.topic_words())
print(f'Guided BERTopic: {len(guided.topic_ids_)} clusters, catálogo com {len(CLASSES)} classes')
print(f'Classes com algum cluster associado: {(guided.projection_.sum(0) > 0).sum()}')
display(guided.mapping_)

## 4. Qual método chegou mais perto das classes originais?

**LRAP é o critério principal:** avalia se as classes verdadeiras aparecem antes das demais. Maior é melhor. Ranking loss e coverage error devem ser menores; coverage mede quantas classes precisamos percorrer no ranking para encontrar todos os rótulos reais.

F1 micro, F1 macro e Jaccard avaliam as classes atribuídas com o `k` escolhido na validação. F1 macro mostra o desempenho médio entre as 54 classes. Compare também rejeições, outliers e suporte das classes raras.

In [ ]:
directions = {'LRAP': True, 'Ranking loss': False, 'Coverage error': False,
              'Acerto top-1': True, 'F1 micro': True, 'F1 macro (54)': True, 'Jaccard': True}
comparison = []
for metric, higher in directions.items():
    values = metrics[metric]
    winner = 'Empate' if np.isclose(values.iloc[0], values.iloc[1]) else (values.idxmax() if higher else values.idxmin())
    comparison.append({'Métrica': metric, 'Melhor direção': 'Maior' if higher else 'Menor', 'Vencedor': winner})
display(pd.DataFrame(comparison))
print(f"Maior LRAP observado no teste: {metrics['LRAP'].idxmax()}.")
print('Este resultado vale para este catálogo e split; não estabelece superioridade geral.')
plot_data = metrics[list(directions)].reset_index().melt(id_vars='Método', var_name='Métrica', value_name='Valor')
fig = px.bar(plot_data, x='Método', y='Valor', color='Método', facet_col='Métrica', facet_col_wrap=3,
             title='Recuperação das classes predefinidas no teste')
fig.update_yaxes(matches=None)
fig.update_layout(height=750, showlegend=False)
fig.show()

In [ ]:
display(per_class.pivot(index='Classe', columns='Método', values='F1').join(support.set_index('Classe')))
px.bar(per_class, x='Classe', y='F1', color='Método', barmode='group',
       hover_data=['Suporte teste', 'Precisão', 'Recall'], title='F1 por classe original', height=550).show()

## 5. Inspecionar exemplos e planejar a próxima rodada

Confira os rótulos reais e atribuídos nos mesmos artigos. Pares de classes com descritores idênticos podem empatar; top-k usa a ordem fixa do catálogo nesses casos. Descrições melhores precisam ser definidas com conhecimento de domínio e avaliadas na validação.

Experimentos sugeridos: variar força das sementes no LDA, revisar o catálogo e variar o tamanho mínimo dos clusters no BERTopic. Preserve o teste para a avaliação final.

Arquivos e parâmetros estão na pasta indicada abaixo. Referência: [Guided BERTopic](https://maartengr.github.io/BERTopic/getting_started/guided/guided.html).

In [ ]:
examples = None
for slug in ['seeded_lda', 'guided_bertopic']:
    predictions = pd.read_csv(OUTPUT / f'{slug}_atribuicoes.csv').fillna('Sem atribuição')
    predictions = predictions.rename(columns={'classes_atribuidas': slug})
    examples = predictions if examples is None else examples.merge(
        predictions[['doc_id', slug]], on='doc_id', validate='one_to_one')
display(examples.head(15))
print(f'Resultados salvos em: {OUTPUT}')